<div dir="rtl">

# הנדסת פיצ'רים: עמודות הרכב

</div>

In [1]:
# ייבוא ספריות, קבוע אקראיות אחד לכל הפרויקט ובדיקת גרסאות
import sys
import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42

In [2]:
# טעינת הדאטה והצגת חמש שורות אקראיות

df = pd.read_csv("dataset.csv")
print(df.shape)
df.sample(5, random_state=RANDOM_STATE)

(45000, 30)


,customer_id_hash,agent_code,country,area_type,region_density,winter_severity,driver_age,gender,marital_status,license_years,...,annual_km_estimated,pct_intercity,pct_night_driving,parking_type,current_premium,claim_count,at_fault,days_to_settlement,police_report_filed,annual_claim_cost
37979,0ebf89f0,A020,FR,suburban,806,3.0,63,M,married,36,...,66500.0,10,45,garage,4270,0,NaN,NaN,NaN,0
9911,e7bedf11,A143,US,rural,24,0.0,38,M,divorced,20,...,17500.0,15,5,garage,1860,0,NaN,NaN,NaN,0
43386,9ff342a8,A060,FR,suburban,1115,3.8,84,M,married,66,...,3000.0,0,10,garage,1950,0,NaN,NaN,NaN,0
13822,b66c36b7,A004,DE,urban,10949,6.6,48,M,divorced,30,...,NaN,20,30,lot,13810,0,NaN,NaN,NaN,0
44810,30fbec1d,A004,US,rural,100,6.4,80,M,married,62,...,15500.0,55,10,garage,5160,0,NaN,NaN,NaN,0


In [3]:
# פונקציות עזר לניקוי: אחת לעמודות מספר ואחת לעמודות טקסט, שעובדות גם על עמודה ריקה לגמרי
def to_num(s):
    # המרה למספר אחרי הסרת פסיקים ורווחים, וערך אינסופי הופך לחסר
    out = pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")
    return out.replace([np.inf, -np.inf], np.nan)

def clean_text(s, case="lower"):
    # הסרת רווחים ואחידות אותיות, וערך חסר או מחרוזת ריקה הופכים לחסר
    out = s.astype(str).str.strip()
    out = out.str.upper() if case == "upper" else out.str.lower()
    return out.where(s.notna() & (out != ""))

In [4]:
# פונקציית הניקוי הסופית של עמודות הנהג ההיסטוריה והגאוגרפיה
def clean_driver_cols(df):
    df = df.copy()

    # מדינה וסוג אזור: איחוד כתיבה, וערך חסר או ריק נשאר חסר
    df["country"] = clean_text(df["country"], case="upper")
    df["area_type"] = clean_text(df["area_type"], case="lower")

    # צפיפות: המרה למספר, וערך שאינו חיובי הופך לחסר
    df["region_density"] = to_num(df["region_density"])
    df.loc[df["region_density"] <= 0, "region_density"] = np.nan

    # חומרת חורף: ערך מחוץ לטווח אפס עד עשר הופך לחסר
    df["winter_severity"] = to_num(df["winter_severity"])
    df.loc[(df["winter_severity"] < 0) | (df["winter_severity"] > 10), "winter_severity"] = np.nan

    # גיל: גיל מתחת לשש עשרה או מעל מאה הופך לחסר
    df["driver_age"] = to_num(df["driver_age"])
    df.loc[(df["driver_age"] < 16) | (df["driver_age"] > 100), "driver_age"] = np.nan

    # מגדר: מיפוי לשני ערכים תקניים, וערך לא מוכר הופך לחסר
    gender_map = {"m": "M", "male": "M", "f": "F", "female": "F"}
    df["gender"] = clean_text(df["gender"], case="lower").map(gender_map)

    # מצב משפחתי: רק ארבעת הערכים החוקיים נשארים, וכל ערך אחר הופך לחסר
    valid_marital = ["single", "married", "divorced", "widowed"]
    df["marital_status"] = clean_text(df["marital_status"], case="lower")
    df.loc[~df["marital_status"].isin(valid_marital), "marital_status"] = np.nan

    # ותק רישיון: ערך שלילי הופך לחסר, וגם ותק שמשמעותו קבלת רישיון לפני גיל שש עשרה
    df["license_years"] = to_num(df["license_years"])
    df.loc[df["license_years"] < 0, "license_years"] = np.nan
    df.loc[df["driver_age"] - df["license_years"] < 16, "license_years"] = np.nan

    # תביעות בעבר: ערך שלילי הופך לחסר וערך מעל שש הופך לשש
    df["prior_claims_5y"] = to_num(df["prior_claims_5y"])
    df.loc[df["prior_claims_5y"] < 0, "prior_claims_5y"] = np.nan
    df["prior_claims_5y"] = df["prior_claims_5y"].clip(upper=6)

    # נהג עם אפס שנות ותק לא יכול היה לצבור תביעות בעבר ולכן ערך חסר אצלו הופך לאפס
    df.loc[(df["license_years"] == 0) & (df["prior_claims_5y"].isna()), "prior_claims_5y"] = 0

    # שנים ללא תביעה: ערך שלילי או גדול מוותק הרישיון הופך לחסר וערך מעל שלושים הופך לשלושים
    df["no_claims_years"] = to_num(df["no_claims_years"])
    df.loc[df["no_claims_years"] < 0, "no_claims_years"] = np.nan
    df.loc[df["no_claims_years"] > df["license_years"], "no_claims_years"] = np.nan
    df["no_claims_years"] = df["no_claims_years"].clip(upper=30)

    # דוחות תנועה: ערך שלילי הופך לחסר
    df["violations_3y"] = to_num(df["violations_3y"])
    df.loc[df["violations_3y"] < 0, "violations_3y"] = np.nan

    return df

In [5]:
# פונקציית הניקוי הסופית של עמודות הרכב והשימוש
def clean_vehicle_cols(df):
    df = df.copy()

    # סוג רכב: איחוד כתיבה, וערך שאינו אחד מששת הסוגים החוקיים הופך לחסר
    valid_vehicle = ["small", "sedan", "suv", "sports", "pickup", "motorcycle"]
    df["vehicle_type"] = clean_text(df["vehicle_type"], case="lower")
    df.loc[~df["vehicle_type"].isin(valid_vehicle), "vehicle_type"] = np.nan

    # גיל רכב: ערך שלילי הופך לחסר, וערך מעל עשרים וחמש הופך לעשרים וחמש
    df["vehicle_age"] = to_num(df["vehicle_age"])
    df.loc[df["vehicle_age"] < 0, "vehicle_age"] = np.nan
    df["vehicle_age"] = df["vehicle_age"].clip(upper=25)

    # עוצמת מנוע: ערך שאינו חיובי הופך לחסר
    df["engine_hp"] = to_num(df["engine_hp"])
    df.loc[df["engine_hp"] <= 0, "engine_hp"] = np.nan

    # שווי רכב: הפסיקים מוסרים בפונקציית העזר, וערך שאינו חיובי הופך לחסר
    df["vehicle_value"] = to_num(df["vehicle_value"])
    df.loc[df["vehicle_value"] <= 0, "vehicle_value"] = np.nan

    # מערכות בטיחות: ערך מחוץ לטווח אפס עד שמונה הופך לחסר
    df["safety_features_count"] = to_num(df["safety_features_count"])
    df.loc[(df["safety_features_count"] < 0) | (df["safety_features_count"] > 8), "safety_features_count"] = np.nan

    # טלמטיקה: ערך שאינו אפס או אחד הופך לחסר
    df["has_telematics"] = to_num(df["has_telematics"])
    df.loc[~df["has_telematics"].isin([0, 1]), "has_telematics"] = np.nan

    # סוג שימוש: איחוד כתיבה, וערך שאינו אחד מחמשת הסוגים החוקיים הופך לחסר
    valid_usage = ["commute", "admin_work", "leisure", "professional", "delivery"]
    df["usage_type"] = clean_text(df["usage_type"], case="lower")
    df.loc[~df["usage_type"].isin(valid_usage), "usage_type"] = np.nan

    # קילומטרים: ערך שאינו חיובי או מעל מאתיים אלף הופך לחסר, כולל הערך המוסווה של תשע מאות תשעים ותשעה אלף
    df["annual_km_estimated"] = to_num(df["annual_km_estimated"])
    df.loc[(df["annual_km_estimated"] <= 0) | (df["annual_km_estimated"] > 200000), "annual_km_estimated"] = np.nan

    # אחוז נסיעה בין עירונית ואחוז נהיגה בלילה: ערך מחוץ לטווח אפס עד מאה הופך לחסר, כל עמודה בנפרד
    for c in ["pct_intercity", "pct_night_driving"]:
        df[c] = to_num(df[c])
        df.loc[(df[c] < 0) | (df[c] > 100), c] = np.nan

    # סוג חניה: איחוד כתיבה, וערך שאינו אחד משלושת הסוגים החוקיים הופך לחסר
    valid_parking = ["garage", "street", "lot"]
    df["parking_type"] = clean_text(df["parking_type"], case="lower")
    df.loc[~df["parking_type"].isin(valid_parking), "parking_type"] = np.nan

    return df

In [6]:
# רשימת עמודות המודל בסדר קבוע, ופונקציית ההכנה שמנקה ומחזירה רק אותן
TARGET = "annual_claim_cost"
FEATURE_COLS = [
    "country", "area_type", "region_density", "winter_severity", "driver_age", "gender", "marital_status",
    "license_years", "prior_claims_5y", "no_claims_years", "violations_3y",
    "vehicle_type", "vehicle_age", "engine_hp", "vehicle_value", "safety_features_count", "has_telematics",
    "usage_type", "annual_km_estimated", "pct_intercity", "pct_night_driving", "parking_type",
]

def prepare_data(df):
    # עמודה חסרה נוספת כריקה, ניקוי שני החלקים, ושמירה רק על עמודות המודל בסדר קבוע
    df = df.copy()
    for c in FEATURE_COLS:
        if c not in df.columns:
            df[c] = np.nan
    df = clean_driver_cols(df)
    df = clean_vehicle_cols(df)
    return df[FEATURE_COLS]

<div dir="rtl">

## פיצ'ר מועמד: לוג של שווי הרכב

שווי הרכב מוטה מאוד ימינה. לפני שנבנה את הלוג, נבדוק אם הקשר בין השווי לעלות נחלש בערכים גבוהים, ואז לוג יעזור, או מתחזק, ואז לוג יפגע.

</div>

In [7]:
# בדיקה האם לוגריתם של שווי הרכב מתאים יותר לקשר עם העלות מאשר השווי עצמו
X = prepare_data(df)
y = df[TARGET]
value = X["vehicle_value"]
print("Skew: raw", round(value.skew(), 2), "| log", round(np.log(value).skew(), 2))
print("Correlation with cost: raw", round(value.corr(y), 3), "| log", round(np.log(value).corr(y), 3))
print(y.groupby(pd.qcut(value, 10)).mean().round(0))

Skew: raw 2.81 | log -0.2
Correlation with cost: raw 0.253 | log 0.229
vehicle_value
(2999.999, 13500.0]      986.0
(13500.0, 20000.0]      1420.0
(20000.0, 27000.0]      1815.0
(27000.0, 35000.0]      2081.0
(35000.0, 44500.0]      2302.0
(44500.0, 55500.0]      2546.0
(55500.0, 70500.0]      3206.0
(70500.0, 90000.0]      3936.0
(90000.0, 123500.0]     4905.0
(123500.0, 700000.0]    7420.0
Name: annual_claim_cost, dtype: float64


<div dir="rtl">

הקורלציה עם העלות ירדה אחרי הלוג (מ-0.253 ל-0.229), והטבלה מראה שהעלות עולה בקצב קבוע לפי השווי עצמו. כלומר, הקשר כבר קרוב לקו ישר, והלוג מעקם אותו. נאמת את זה ב- CV.

</div>

In [8]:
# השוואה בתיקוף צולב בין שווי הרכב הגולמי, הלוג שלו ושניהם יחד
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression

kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
options = {"raw value": X[["vehicle_value"]],
           "log value": np.log(X[["vehicle_value"]]),
           "raw + log": pd.concat([X["vehicle_value"], np.log(X["vehicle_value"]).rename("log_value")], axis=1)}
for name, data in options.items():
    rmse = -cross_val_score(LinearRegression(), data, y, cv=kf, scoring="neg_root_mean_squared_error")
    print(f"{name:10} RMSE {rmse.mean():8.1f} | std {rmse.std():5.1f}")

raw value  RMSE   7296.9 | std 407.8
log value  RMSE   7339.7 | std 428.3
raw + log  RMSE   7295.5 | std 409.7


<div dir="rtl">

### מסקנה לגבי לוג שווי הרכב: נדחה

- **ההשערה:** שווי הרכב מוטה ימינה (skew 2.81), ולוג יקרב את הקשר לקו ישר.
- **מה מצאנו:** הלוג אכן יישר את ההתפלגות (skew 0.2-), אבל הקורלציה עם העלות ירדה מ-0.253 ל-0.229. לפי עשירוני השווי, העלות עולה בקצב קבוע של בערך 25 עד 60 ₪ לכל 1,000 ₪ של שווי, כלומר הקשר כבר ישר. הלוג מכווץ את הרכבים היקרים ולכן מעקם אותו.
- **תיקוף צולב:** הלוג לבדו גרוע ב-42.8. לוג לצד השווי הגולמי שיפר רק ב-1.4, הרבה פחות מהפיזור בין ה-folds (408).
- **החלטה:** השווי נכנס למודל כמו שהוא, בלי לוג. לוג עוזר כשההשפעה נחלשת בערכים גבוהים, כמו בצפיפות, ולא כשהיא קבועה.

</div>

<div dir="rtl">

## פיצ'ר מועמד: לוג של הק"מ השנתי

אותה בדיקה כמו בשווי הרכב: האם הקשר בין הק"מ לעלות נחלש בערכים גבוהים, ואז לוג יעזור, או קבוע, ואז לוג יפגע.

</div>

In [9]:
# בדיקה האם לוגריתם של הקילומטרים מתאים יותר לקשר עם העלות מאשר הקילומטרים עצמם
km = X["annual_km_estimated"]
print("Skew: raw", round(km.skew(), 2), "| log", round(np.log(km).skew(), 2))
print("Correlation with cost: raw", round(km.corr(y), 3), "| log", round(np.log(km).corr(y), 3))
print(y.groupby(pd.qcut(km, 10)).mean().round(0))

Skew: raw 2.79 | log -0.04
Correlation with cost: raw 0.209 | log 0.187
annual_km_estimated
(499.999, 3500.0]      1177.0
(3500.0, 5000.0]       1661.0
(5000.0, 6500.0]       1896.0
(6500.0, 8000.0]       1967.0
(8000.0, 9500.0]       2326.0
(9500.0, 11500.0]      2683.0
(11500.0, 13500.0]     2904.0
(13500.0, 17000.0]     3134.0
(17000.0, 23000.0]     3989.0
(23000.0, 120000.0]    6099.0
Name: annual_claim_cost, dtype: float64


In [10]:
# השוואה בתיקוף צולב בין הקילומטרים הגולמיים הלוג שלהם ושניהם יחד עם השלמת חסרים בתוך כל קיפול
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

options = {"raw km": X[["annual_km_estimated"]],
           "log km": np.log(X[["annual_km_estimated"]]),
           "raw + log": pd.concat([km, np.log(km).rename("log_km")], axis=1)}
for name, data in options.items():
    model = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), LinearRegression())
    rmse = -cross_val_score(model, data, y, cv=kf, scoring="neg_root_mean_squared_error")
    print(f"{name:10} RMSE {rmse.mean():8.1f} | std {rmse.std():5.1f}")

raw km     RMSE   7372.0 | std 438.7
log km     RMSE   7396.4 | std 441.1
raw + log  RMSE   7372.3 | std 438.5


<div dir="rtl">

### מסקנה לגבי לוג הק"מ: נדחה

- **ההשערה:** הק"מ מוטה ימינה (skew 2.79), ולוג יקרב את הקשר לקו ישר.
- **מה מצאנו:** הלוג יישר את ההתפלגות (skew 0.04-), אבל הקורלציה עם העלות ירדה מ-0.209 ל-0.187. העלות עולה לאורך כל הטווח, ובעשירון העליון אף קופצת מ-3,989 ל-6,099, כך שהלוג מכווץ דווקא את האזור החשוב.
- **תיקוף צולב (עם השלמת חסרים בתוך כל fold):** הלוג לבדו גרוע ב-24.4, ולוג לצד הק"מ הגולמי לא שיפר בכלל.
- **החלטה:** הק"מ נכנס למודל כמו שהוא. החוסר בק"מ, שמנבא עלות פי 1.9, נשמר בעזרת עמודת הסימון של ה-Imputer.

</div>

<div dir="rtl">

## פיצ'ר מועמד: דגל שימוש לעבודה

דגל של 1 לנהגים במשלוחים או בנהיגה מקצועית. נבדוק אם הוא מוסיף מידע מעבר לעמודות של סוג השימוש, שה-OneHotEncoder כבר יוצר.

</div>

In [11]:
# בדיקה האם דגל שימוש לעבודה מוסיף מידע מעבר לעמודות של סוג השימוש שהקידוד כבר יוצר
usage = pd.get_dummies(X["usage_type"], dtype=float)
is_work = X["usage_type"].isin(["delivery", "professional"]).astype(float)
print("Average cost: other", round(y[is_work == 0].mean()), "| work", round(y[is_work == 1].mean()))

usage_with_flag = usage.copy()
usage_with_flag["is_work"] = is_work

rmse_without = -cross_val_score(LinearRegression(), usage, y, cv=kf, scoring="neg_root_mean_squared_error")
rmse_with = -cross_val_score(LinearRegression(), usage_with_flag, y, cv=kf, scoring="neg_root_mean_squared_error")
print("Without flag: RMSE", round(rmse_without.mean(), 1))
print("With flag:    RMSE", round(rmse_with.mean(), 1))
print("Flag equals sum of two columns:", (is_work == usage["delivery"] + usage["professional"]).all())

Average cost: other 2555 | work 6330
Without flag: RMSE 7422.5
With flag:    RMSE 7422.5
Flag equals sum of two columns: True


<div dir="rtl">

### מסקנה לגבי דגל שימוש לעבודה: נדחה

- **ההשערה:** נהגים במשלוחים ובנהיגה מקצועית יקרים יותר, 6,330 לעומת 2,555 בממוצע.
- **מה מצאנו:** הדגל שווה בדיוק לסכום של שתי עמודות שה-OneHotEncoder כבר יוצר מסוג השימוש. מודל לינארי יכול לבנות סכום בעצמו, ולכן הדגל לא מוסיף מידע.
- **תיקוף צולב:** RMSE של 7,422.5 עם הדגל ובלעדיו, בלי שום הבדל.
- **החלטה:** הדגל לא נכנס. ההבדל בין סוגי השימוש כבר נלמד מהקידוד, ומה שמודל לינארי לא יכול לבנות לבד הוא מכפלה, ולכן הפיצ'ר הבא הוא אינטראקציה.

</div>

<div dir="rtl">

## פיצ'ר מועמד: שווי הרכב כפול שימוש לעבודה (אינטראקציה)

מודל לינארי נותן לשווי הרכב מקדם אחד לכל הנהגים. המכפלה מאפשרת לו לתת לשווי השפעה חזקה יותר אצל נהגים במשלוחים ובנהיגה מקצועית. נבדוק אם הפער בעלות בין נהגי עבודה לשאר גדל ככל שהשווי עולה.

</div>

In [12]:
# עלות ממוצעת לפי חמש קבוצות של שווי הרכב בנפרד לנהגי עבודה ולשאר הנהגים
value_group = pd.qcut(X["vehicle_value"], 5)
pd.crosstab(value_group, is_work, values=y, aggfunc="mean").round(0)

usage_type,0.0,1.0
vehicle_value,,
"(2999.999, 20000.0]",936.0,2708.0
"(20000.0, 35000.0]",1603.0,4081.0
"(35000.0, 55500.0]",2061.0,5081.0
"(55500.0, 90000.0]",2900.0,7966.0
"(90000.0, 700000.0]",5238.0,12918.0


In [13]:
# בדיקה בתיקוף צולב האם המכפלה של שווי הרכב בשימוש לעבודה משפרת מעבר לשווי ולסוג השימוש
base = usage.copy()
base["vehicle_value"] = X["vehicle_value"]

with_interaction = base.copy()
with_interaction["value_x_work"] = X["vehicle_value"] * is_work

rmse_base = -cross_val_score(LinearRegression(), base, y, cv=kf, scoring="neg_root_mean_squared_error")
rmse_inter = -cross_val_score(LinearRegression(), with_interaction, y, cv=kf, scoring="neg_root_mean_squared_error")
print("Value and usage:     RMSE", round(rmse_base.mean(), 1), "| std", round(rmse_base.std(), 1))
print("With interaction:    RMSE", round(rmse_inter.mean(), 1), "| std", round(rmse_inter.std(), 1))

Value and usage:     RMSE 7164.2 | std 398.7
With interaction:    RMSE 7119.6 | std 387.5


In [14]:
# בדיקה בכמה קיפולים המודל עם המכפלה טוב יותר מהמודל בלעדיה
improvement = rmse_base - rmse_inter
print("Improvement per fold:", improvement.round(1))
print("Folds improved:", (improvement > 0).sum(), "of 10")

Improvement per fold: [ 52.4  72.7  56.   45.7  67.3  41.2  28.9  99.4 -39.3  21. ]
Folds improved: 9 of 10


<div dir="rtl">

### מסקנה לגבי שווי הרכב כפול שימוש לעבודה: נכנס

- **נוסחה:** value_x_work = שווי הרכב כפול דגל שימוש לעבודה (1 למשלוחים ולנהיגה מקצועית). זו אינטראקציה, פיצ'ר נומרי מורכב.
- **מוטיבציה:** הפער בעלות בין נהגי עבודה לשאר גדל עם שווי הרכב: 1,772 ברכבים עד 20,000, ו-7,680 ברכבים מעל 90,000, פי 4.3. היחס נשאר בערך קבוע, פי 2.5 עד 2.9, כלומר העבודה מכפילה את העלות ולא מוסיפה סכום קבוע. בלי המכפלה, מודל לינארי נותן פער קבוע.
- **תיקוף צולב:** ה-RMSE ירד ב-44.6, והשיפור הופיע ב-9 מתוך 10 folds.
- **הסתייגות:** השיפור קטן מהפיזור בין ה-folds (398.7), אבל הפיזור הזה משותף לשני המודלים, כי הם נבדקו על אותם folds. ה-ElasticNet יאפס את המקדם אם הפיצ'ר לא יתרום במודל המלא.

</div>

<div dir="rtl">

## פיצ'ר מועמד: ק"מ כפול שימוש לעבודה (אינטראקציה)

אותו רעיון כמו בשווי הרכב: נבדוק אם ההשפעה של הק"מ חזקה יותר אצל נהגים במשלוחים ובנהיגה מקצועית.

</div>

In [15]:
# עלות ממוצעת לפי חמש קבוצות של קילומטרים בנפרד לנהגי עבודה ולשאר הנהגים
km_group = pd.qcut(X["annual_km_estimated"], 5)
pd.crosstab(km_group, is_work, values=y, aggfunc="mean").round(0)

usage_type,0.0,1.0
annual_km_estimated,,
"(499.999, 5000.0]",1420.0,NaN
"(5000.0, 8000.0]",1929.0,2554.0
"(8000.0, 11500.0]",2517.0,2638.0
"(11500.0, 17000.0]",3035.0,2924.0
"(17000.0, 120000.0]",4519.0,6116.0


In [16]:
# בדיקה בתיקוף צולב האם המכפלה של הקילומטרים בשימוש לעבודה משפרת מעבר לקילומטרים ולסוג השימוש
base_km = usage.copy()
base_km["annual_km_estimated"] = X["annual_km_estimated"]

with_km_interaction = base_km.copy()
with_km_interaction["km_x_work"] = X["annual_km_estimated"] * is_work

model = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), LinearRegression())
rmse_km_base = -cross_val_score(model, base_km, y, cv=kf, scoring="neg_root_mean_squared_error")
rmse_km_inter = -cross_val_score(model, with_km_interaction, y, cv=kf, scoring="neg_root_mean_squared_error")
print("Km and usage:        RMSE", round(rmse_km_base.mean(), 1))
print("With interaction:    RMSE", round(rmse_km_inter.mean(), 1))
print("Folds improved:", (rmse_km_base > rmse_km_inter).sum(), "of 10")

Km and usage:        RMSE 7357.1
With interaction:    RMSE 7344.7
Folds improved: 9 of 10


<div dir="rtl">

### מסקנה לגבי ק"מ כפול שימוש לעבודה: נדחה

- **ההשערה:** ההשפעה של כל ק"מ חזקה יותר אצל נהגי עבודה.
- **מה מצאנו:** הפער בין נהגי עבודה לשאר לא גדל באופן עקבי (625, 121, 111-, 1,597). בקבוצות האמצעיות יש רק 24 ו-61 נהגי עבודה, ולכן הממוצעים לא אמינים. הקפיצה בקבוצה העליונה נובעת מכך שבתוכה נהגי העבודה נוסעים יותר (חציון 31,500 לעומת 21,500), ולא מהשפעה חזקה יותר לכל ק"מ.
- **בעיה נוספת:** 48.6% מנהגי העבודה לא דיווחו ק"מ (לעומת 7.2% מהשאר), ולכן אצל כמעט חצי מהם המכפלה חסרה.
- **תיקוף צולב:** שיפור של 12.4 בלבד.
- **החלטה:** הפיצ'ר לא נכנס.

</div>

<div dir="rtl">

## פיצ'ר מועמד: נהג צעיר כפול עוצמת מנוע (אינטראקציה)

ההשערה: לנהג מנוסה, עוד כוחות סוס מוסיפים מעט סיכון, ולנהג צעיר הרבה. נהג צעיר מוגדר כמו אצל ליעד, עד גיל 21 כולל. נבדוק אם הפער בעלות בין צעירים לשאר גדל עם עוצמת המנוע, וגם כמה נהגים יש בכל קבוצה.

</div>

In [17]:
# עלות ממוצעת ומספר נהגים לפי חמש קבוצות של עוצמת מנוע בנפרד לנהגים צעירים ולשאר
is_young = (X["driver_age"] <= 21).astype(float).rename("is_young")
hp_group = pd.qcut(X["engine_hp"], 5)
print(pd.crosstab(hp_group, is_young, values=y, aggfunc="mean").round(0), "\n")
print(pd.crosstab(hp_group, is_young))

is_young           0.0      1.0
engine_hp                      
(11.999, 87.0]  1208.0   2224.0
(87.0, 119.0]   1624.0   3797.0
(119.0, 153.0]  2587.0   4135.0
(153.0, 189.0]  3389.0   8052.0
(189.0, 650.0]  5723.0  15389.0 

is_young         0.0  1.0
engine_hp                
(11.999, 87.0]  8617  460
(87.0, 119.0]   8687  370
(119.0, 153.0]  8757  302
(153.0, 189.0]  8644  266
(189.0, 650.0]  8474  423


In [18]:
# בדיקה בתיקוף צולב האם המכפלה של נהג צעיר בעוצמת המנוע משפרת מעבר לשני הפיצ'רים לבד
base_hp = pd.DataFrame({"engine_hp": X["engine_hp"], "is_young": is_young})

with_hp_interaction = base_hp.copy()
with_hp_interaction["young_x_hp"] = X["engine_hp"] * is_young

rmse_hp_base = -cross_val_score(LinearRegression(), base_hp, y, cv=kf, scoring="neg_root_mean_squared_error")
rmse_hp_inter = -cross_val_score(LinearRegression(), with_hp_interaction, y, cv=kf, scoring="neg_root_mean_squared_error")
print("Hp and young:        RMSE", round(rmse_hp_base.mean(), 1))
print("With interaction:    RMSE", round(rmse_hp_inter.mean(), 1))
print("Folds improved:", (rmse_hp_base > rmse_hp_inter).sum(), "of 10")

Hp and young:        RMSE 7257.3
With interaction:    RMSE 7225.3
Folds improved: 9 of 10


<div dir="rtl">

### מסקנה לגבי נהג צעיר כפול עוצמת מנוע: נכנס

- **נוסחה:** young_x_hp = עוצמת המנוע כפול דגל נהג צעיר (עד גיל 21 כולל, כמו is_young של ליעד). זו אינטראקציה, פיצ'ר נומרי מורכב.
- **מוטיבציה:** הפער בעלות בין צעירים לשאר גדל עם עוצמת המנוע: 1,016 במנועים עד 87 כוחות סוס, ו-9,666 במנועים מעל 189, פי 9.5. ברכב חזק נהג צעיר עולה פי 2.7 מנהג מנוסה, וברכב חלש פי 1.8.
- **אמינות:** בכל קבוצה יש בין 266 ל-460 נהגים צעירים. הפערים בקבוצות העליונות גדולים בהרבה מהשונות האפשרית בממוצע.
- **תיקוף צולב:** ה-RMSE ירד ב-32.0, והשיפור הופיע ב-9 מתוך 10 folds.
- **תיאום:** הפיצ'ר משתמש בגיל הנהג ובהגדרה של ליעד לנהג צעיר.

</div>

<div dir="rtl">

## הפונקציה הסופית: add_vehicle_features

מכילה את שני הפיצ'רים שעברו את הבדיקות: שווי הרכב כפול שימוש לעבודה, ועוצמת המנוע כפול נהג צעיר. מקבלת טבלה אחרי הניקוי, מוסיפה שתי עמודות בסדר קבוע, ולא משנה את העמודות הקיימות.

אם אחת מעמודות המקור חסרה, גם הפיצ'ר חסר, וה-Pipeline משלים אותו. הפונקציה תיכנס לתוך prepare_data יחד עם add_driver_features של ליעד.

</div>

In [19]:
# פונקציה שמוסיפה שתי עמודות חדשות לטבלה אחרי הניקוי בלי לשנות את העמודות הקיימות
VEHICLE_FEATURES = ["value_x_work", "young_x_hp"]

def add_vehicle_features(df):
    df = df.copy()
    usage = df["usage_type"]
    age = df["driver_age"]

    # שימוש לעבודה: אחד למשלוחים ולנהיגה מקצועית, ואם סוג השימוש חסר גם הדגל חסר
    is_work = usage.isin(["delivery", "professional"]).astype(float).where(usage.notna())

    # שווי הרכב כפול שימוש לעבודה, וחסר אם אחד מהם חסר
    df["value_x_work"] = df["vehicle_value"] * is_work

    # נהג צעיר עד גיל עשרים ואחת כולל, ואם הגיל חסר גם הדגל חסר
    is_young = (age <= 21).astype(float).where(age.notna())

    # עוצמת המנוע כפול נהג צעיר, וחסר אם אחד מהם חסר
    df["young_x_hp"] = df["engine_hp"] * is_young

    return df

In [21]:
# בדיקות קיצון לפונקציה: ניסיון לשבור אותה בקלטים שעלולים להגיע בקובץ הבדיקה
import io
def run(d):
    return add_vehicle_features(clean_vehicle_cols(clean_driver_cols(d)))

full = run(df)
base = clean_vehicle_cols(clean_driver_cols(df))
checks = {}

# שתי עמודות חדשות בסדר קבוע והעמודות המקוריות לא השתנו
checks["two new columns in fixed order"] = list(full.columns[-2:]) == VEHICLE_FEATURES and full.shape[1] == base.shape[1] + 2
checks["original columns unchanged"] = full[base.columns].equals(base)

# אין ערכים חסרים חדשים, כי בדאטה אין חסרים בעמודות המקור
checks["no missing values"] = full[VEHICLE_FEATURES].isna().sum().sum() == 0

# מקרי קצה: גיל עשרים ואחת מול עשרים ושתיים, שימוש לעבודה מול פנאי, וערכים חסרים
edge = df.iloc[[0, 0, 0, 0]].copy().reset_index(drop=True)
edge["driver_age"] = [21, 22, np.nan, 30]
edge["usage_type"] = ["delivery", "leisure", "professional", np.nan]
edge["vehicle_value"] = [10000, 10000, np.nan, 10000]
edge["engine_hp"] = [100, 100, 100, 100]
e = run(edge)
checks["edge: work gives value, leisure gives 0"] = e.loc[0, "value_x_work"] == 10000 and e.loc[1, "value_x_work"] == 0
checks["edge: age 21 gives hp, age 22 gives 0"] = e.loc[0, "young_x_hp"] == 100 and e.loc[1, "young_x_hp"] == 0
checks["edge: missing age or value gives missing"] = pd.isna(e.loc[2, "young_x_hp"]) and pd.isna(e.loc[2, "value_x_work"])
checks["edge: missing usage gives missing"] = pd.isna(e.loc[3, "value_x_work"])

# אלף וחמש מאות שורות בודדות זהות לתוצאה על כל הדאטה
idx = df.sample(1500, random_state=RANDOM_STATE).index
checks["1500 single rows match full run"] = all(run(df.loc[[i]]).equals(full.loc[[i]]) for i in idx)

# שורה ריקה מקובץ, ערכים מלוכלכים ואפס שורות
empty_csv = ",".join(df.columns) + "\n" + ",".join([""] * len(df.columns)) + "\n"
checks["empty row from file"] = bool(run(pd.read_csv(io.StringIO(empty_csv)))[VEHICLE_FEATURES].isna().all(axis=None))
dirty = df.iloc[:5].copy().astype(object)
for c in ["driver_age", "usage_type", "vehicle_value", "engine_hp"]:
    dirty[c] = ["", " ", "inf", pd.NA, "1,200"]
try:
    run(dirty)
    checks["dirty values do not crash"] = True
except Exception as ex:
    checks["dirty values do not crash"] = False
    print(type(ex).__name__, ex)
checks["zero rows"] = run(df.iloc[0:0]).shape[0] == 0

# הקלט לא משתנה ושתי הרצות זהות
before = df.copy()
run(before)
checks["input unchanged"] = before.equals(df)
checks["deterministic"] = run(df).equals(full)

for name, passed in checks.items():
    print("OK  " if passed else "FAIL", name)

OK   two new columns in fixed order
OK   original columns unchanged
OK   no missing values
OK   edge: work gives value, leisure gives 0
OK   edge: age 21 gives hp, age 22 gives 0
OK   edge: missing age or value gives missing
OK   edge: missing usage gives missing
OK   1500 single rows match full run
OK   empty row from file
OK   dirty values do not crash
OK   zero rows
OK   input unchanged
OK   deterministic


<div dir="rtl">

### סיכום: פיצ'רי הרכב

| פיצ'ר | נוסחה | סוג | מוטיבציה |
|---|---|---|---|
| value_x_work | שווי הרכב כפול דגל שימוש לעבודה (משלוחים או נהיגה מקצועית) | אינטראקציה (נומרי מורכב) | הפער בין נהגי עבודה לשאר גדל עם השווי, מ-1,772 ברכבים עד 20,000 ל-7,680 מעל 90,000 (פי 4.3). העבודה מכפילה את העלות פי 2.5 עד 2.9, ומודל לינארי בלי מכפלה נותן פער קבוע |
| young_x_hp | עוצמת המנוע כפול דגל נהג צעיר (עד גיל 21, כמו אצל ליעד) | אינטראקציה (נומרי מורכב) | הפער בין צעירים לשאר גדל עם עוצמת המנוע, מ-1,016 במנועים חלשים ל-9,666 במנועים חזקים (פי 9.5). בכל קבוצה יש 266 עד 460 צעירים |

**נבדקו ונדחו:**
- **לוג שווי הרכב:** העלות עולה בקצב קבוע לפי השווי עצמו, והלוג מעקם את הקשר. הקורלציה ירדה מ-0.253 ל-0.229, וב-CV הלוג גרוע ב-42.8.
- **לוג הק"מ:** אותו דפוס. הקורלציה ירדה מ-0.209 ל-0.187, וב-CV הלוג גרוע ב-24.4.
- **דגל שימוש לעבודה:** שווה בדיוק לסכום של שתי עמודות שהקידוד של סוג השימוש כבר יוצר, ולכן אפס שיפור. מודל לינארי יכול לבנות סכום בעצמו.
- **ק"מ כפול שימוש לעבודה:** הפער לא גדל באופן עקבי, בקבוצות האמצעיות יש רק 24 ו-61 נהגי עבודה, ו-48.6% מנהגי העבודה לא דיווחו ק"מ. שיפור של 12.4 בלבד.

**תיקוף צולב:** כל פיצ'ר נבדק לבד, מול מודל לינארי עם עמודות המקור שלו, ב-10 folds. value_x_work הוריד את ה-RMSE ב-44.6, ו-young_x_hp ב-32.0. בשניהם השיפור הופיע ב-9 מתוך 10 folds.

**בדיקות:** 13 בדיקות קיצון עברו, כולל מקרי קצה של גיל 21 מול 22, שימוש לעבודה מול פנאי, וערך חסר שנותן פיצ'ר חסר.

</div>